# 01 HCC raw count와 환자 분할

무엇을 하는가: 준비된 64세포 subset의 count·annotation·환자 분할을 직접 확인한다. 원본 전체를 다시 다운로드하지 않는다.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/paths.yaml').exists()
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', MKL_NUM_THREADS='8', TOKENIZERS_PARALLELISM='false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import yaml, pandas as pd, numpy as np
PATHS = yaml.safe_load((ROOT / 'configs/paths.yaml').read_text())
DATA = Path(PATHS['data_root'])
def run_script(name, config=None, model='scgpt', args=()):
    cmd = [str(ROOT / f'.venvs/vcell-{model}/bin/python'), '-u', str(ROOT / 'scripts' / name)]
    if config: cmd += ['--config', str(ROOT / 'configs' / config)]
    cmd += list(args)
    print(' '.join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout[-16000:])
    if p.returncode:
        print(p.stderr[-16000:])
        p.check_returncode()
    return p
print('code:', ROOT, '\ndata:', DATA)


code: /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver 
data: /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver


In [2]:
import anndata as ad
a = ad.read_h5ad(DATA / 'subsets/smoke.h5ad')
print(a)
print(a.obs[['patient_id','sample_id','tissue_site','cell_type_original','split','n_counts','n_genes','pct_counts_mt']].head().to_string())
print(pd.crosstab(a.obs.patient_id, a.obs.split))

AnnData object with n_obs × n_vars = 64 × 25712
    obs: 'sample_id', 'res.3', 'tissue_site', 'patient_id', 'stage', 'virus', 'cell_type_original', 'cell_id', 'cell_type_coarse', 'split', 'n_counts', 'n_genes', 'pct_counts_mt'
    var: 'gene_id_original', 'gene_symbol', 'ensembl_id', 'id_mapping_status'
    uns: 'dataset_accession', 'qc_config', 'schema_mapping', 'source_manifest'
    layers: 'counts'
                        patient_id sample_id tissue_site cell_type_original  split  n_counts  n_genes  pct_counts_mt
Cell                                                                                                                
HCC01T_AATCGGTGTTGATTCG      HCC01    HCC01T       Tumor                  B  train      6844     1665       2.659264
HCC01T_CGCTGGAGTTCTGAAC      HCC01    HCC01T       Tumor            Myeloid  train      4665     1612      11.468382
HCC01T_CTCATTAGTTCAGACT      HCC01    HCC01T       Tumor         Fibroblast  train      8940     2495       6.957494
HCC01T_GCG

In [3]:
from scipy import sparse
assert sparse.issparse(a.layers['counts'])
assert (a.layers['counts'].data >= 0).all()
assert np.equal(a.layers['counts'].data, np.floor(a.layers['counts'].data)).all()
assert np.array_equal(np.asarray(a.layers['counts'].sum(1)).ravel(), a.obs.n_counts)
for p, rows in a.obs.groupby('patient_id', observed=True): assert rows.split.nunique() == 1
print('count/환자 누수 검사 통과')
print(a.var.id_mapping_status.value_counts())
print(json.loads((ROOT / 'reports/data_audit.json').read_text()))

count/환자 누수 검사 통과
id_mapping_status
unique       18291
unmapped      7365
ambiguous       56
Name: count, dtype: int64
{'matrix_shape': [71915, 25712], 'orientation': 'gene x cell in source; cell x gene in AnnData', 'raw_nonnegative_integers': True, 'metadata_missing_from_matrix': 0, 'matrix_missing_from_metadata': 0, 'duplicate_cell_ids': 0, 'duplicate_gene_symbols': 0, 'ensembl_versions_removed': 'GENCODE gene_id suffix stripped; source IDs are symbols', 'gene_mapping': {'unique': 18291, 'unmapped': 7365, 'ambiguous': 56}, 'count_sum_before_gene_filter': 705606794, 'count_sum_after_gene_filter': 705606794, 'filter': 'no gene filtering in canonical counts', 'wall_seconds': 531.2497341036797, 'subsets': {'comparison': {'shape': [10000, 25712], 'patients': 10, 'split_counts': {'train': 6436, 'val': 2324, 'test': 1240}}, 'pilot': {'shape': [2000, 25712], 'patients': 10, 'split_counts': {'train': 1285, 'val': 462, 'test': 253}}, 'smoke': {'shape': [64, 25712], 'patients': 10, 'split_count

생물학적 해석: 원본 celltype의 T/NK, B, Myeloid, Endothelial, Fibroblast, Hepatocyte를 유지한다. Tumor 조직 출처를 malignant 세포 정답으로 바꾸지 않는다. Normal은 인접 조직이다.

실패 해결: 파일이 없으면 scripts/download_hcc.sh 후 scripts/01_prepare_hcc.py --config configs/hcc_gse149614.yaml을 실행한다. 이 작업은 전체 데이터를 다루며 작은 실습보다 오래 걸린다. sparse count를 전체 dense로 만들지 않는다. 모호한 symbol은 임의 Ensembl ID로 채우지 않는다.